# Gemma4Good Kaggle Submission Evidence Notebook

This notebook is a Kaggle-ready companion artifact for the Gemma 4 Good Hackathon writeup. It is structured around the live Kaggle criteria checked on 2026-05-18: submitted Kaggle Writeup, public video, public source-of-truth code, live demo, cover image, selected track, and a writeup under 1,500 words.

The notebook is not required by Kaggle to run the live server. It should execute as standalone evidence. Any live API/server check in this notebook is optional and skipped by default.

Competition: https://www.kaggle.com/competitions/gemma-4-good-hackathon

Project demo: https://gemma4good.home.complete.tech/menu/fieldreport/

Primary technical claim: a local-first field reporting workflow uses Gemma 4 through a llama.cpp Vulkan runtime to turn images, location, and notes into structured, reviewable civic action reports.

## Kaggle Compliance Checklist

- [ ] Final Kaggle Writeup submitted before 2026-05-18 23:59 UTC.
- [ ] Track selected on the writeup.
- [ ] Writeup is 1,500 words or less.
- [ ] YouTube video is public, directly linked, login-free, and 3 minutes or less.
- [ ] Public code repository or public Kaggle Notebook is linked under Project Links.
- [ ] Live demo URL or demo files are attached.
- [ ] Media Gallery includes a required cover image.
- [ ] Writeup explains architecture, exact Gemma 4 usage, challenges, and technical choices.
- [ ] Notebook runs without requiring private/live server access; live API checks are optional.
- [ ] Any training data, generated data, or model weights are clearly documented and publishable.

Scoring alignment: Impact & Vision 40 points, Video Pitch & Storytelling 30 points, Technical Depth & Execution 30 points.

In [ ]:
from pathlib import Path
import json
import os
import platform
import sys

NOTEBOOK_DIR = Path.cwd()

def find_project_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "DOMAIN_CONTEXT.md").exists() and (path / "api").exists():
            return path
    return start

PROJECT_ROOT = find_project_root(NOTEBOOK_DIR)
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Notebook directory:", NOTEBOOK_DIR)
print("Project root:", PROJECT_ROOT)

In [ ]:
submission_links = {
    "competition": "https://www.kaggle.com/competitions/gemma-4-good-hackathon",
    "live_demo": os.environ.get("GEMMA4GOOD_DEMO_URL", "https://gemma4good.home.complete.tech/menu/fieldreport/"),
    "public_code": os.environ.get("GEMMA4GOOD_PUBLIC_CODE_URL", "TODO: add public GitHub or Kaggle Notebook URL"),
    "video": os.environ.get("GEMMA4GOOD_VIDEO_URL", "TODO: add public YouTube URL"),
    "writeup": os.environ.get("GEMMA4GOOD_WRITEUP_URL", "TODO: add Kaggle Writeup URL after creation"),
}
print(json.dumps(submission_links, indent=2))

## Architecture Evidence

The public app should be demonstrated as the working product, while this notebook verifies the implementation surface that supports the demo:

- Browser page: `public/menu/fieldreport/index.html`
- API server: `api/server.mjs`
- System prompt: `api/prompts/fieldreport-system.md`
- Request fixture: `api/fixtures/sample-fieldreport-request.json`
- Analysis schema: `docs/schemas/fieldreport-analysis.schema.json`
- Reviewed report schema: `docs/schemas/reviewed-fieldreport.schema.json`

Deployed runtime to describe in the writeup: Gemma 4 26B GGUF `Q4_K_M` served by llama.cpp with Vulkan on local hardware, exposed privately to the same-origin Field Report API.

In [ ]:
evidence_paths = [
    "public/menu/fieldreport/index.html",
    "api/server.mjs",
    "api/prompts/fieldreport-system.md",
    "api/fixtures/sample-fieldreport-request.json",
    "docs/schemas/fieldreport-analysis.schema.json",
    "docs/schemas/reviewed-fieldreport.schema.json",
    "docs/kaggle-gemma4good-criteria.md",
]

rows = []
for rel in evidence_paths:
    path = PROJECT_ROOT / rel
    rows.append({"path": rel, "exists": path.exists(), "size_bytes": path.stat().st_size if path.exists() else None})

for row in rows:
    status = "OK" if row["exists"] else "MISSING"
    print(f"{status:7} {row['path']} {row['size_bytes'] or ''}")

In [ ]:
fixture_path = PROJECT_ROOT / "api/fixtures/sample-fieldreport-request.json"
if fixture_path.exists():
    sample_request = json.loads(fixture_path.read_text())
else:
    sample_request = {
        "imageDataUrl": "data:image/png;base64,...",
        "tag": "water_quality",
        "district": "Hodan",
        "city": "Mogadishu",
        "note": "Long queue and discolored water near a damaged pipe.",
        "lat": "2.0469",
        "lon": "45.3182",
    }

redacted = dict(sample_request)
if "imageDataUrl" in redacted:
    redacted["imageDataUrl"] = redacted["imageDataUrl"][:32] + "..."
print(json.dumps(redacted, indent=2))

In [ ]:
schema_paths = [
    PROJECT_ROOT / "docs/schemas/fieldreport-analysis.schema.json",
    PROJECT_ROOT / "docs/schemas/reviewed-fieldreport.schema.json",
]

for path in schema_paths:
    print("\n==", path.name, "==")
    if not path.exists():
        print("missing")
        continue
    schema = json.loads(path.read_text())
    print("title:", schema.get("title"))
    print("required:", schema.get("required", []))
    print("properties:", sorted(schema.get("properties", {}).keys()))

## Demo Image Dataset Status

The `demo-images` data can support examples and later fine-tuning only after QA. If metadata still says `pending_regeneration` or `needs_review`, do not present it as final approved training data in the Kaggle writeup.

In [ ]:
from collections import Counter

demo_root = PROJECT_ROOT / "demo-images"
json_files = sorted(demo_root.glob("**/*.json")) if demo_root.exists() else []
status_counts = Counter()
quality_counts = Counter()
category_counts = Counter()

for path in json_files:
    try:
        record = json.loads(path.read_text())
    except Exception as exc:
        status_counts[f"invalid_json:{type(exc).__name__}"] += 1
        continue
    status_counts[record.get("generation_status", "missing")] += 1
    quality_counts[record.get("quality_status", "missing")] += 1
    category_counts[path.parent.name] += 1

print("records:", len(json_files))
print("generation_status:", dict(status_counts))
print("quality_status:", dict(quality_counts))
print("categories:", dict(category_counts.most_common(20)))

## Optional Live API Smoke Test

This cell is intentionally opt-in. Set `GEMMA4GOOD_API_BASE` to a public or reachable base URL before running it. On Kaggle, external access may depend on notebook settings and the public demo availability.

In [ ]:
import urllib.error
import urllib.request

api_base = os.environ.get("GEMMA4GOOD_API_BASE", "").rstrip("/")
if not api_base:
    print("Skipped. Set GEMMA4GOOD_API_BASE to run the live API check.")
else:
    url = f"{api_base}/api/fieldreport/health"
    try:
        with urllib.request.urlopen(url, timeout=20) as response:
            print(response.status, response.read().decode("utf-8")[:1000])
    except urllib.error.URLError as exc:
        print("API check failed:", exc)

## Writeup Outline Under 1,500 Words

1. Problem: communities need low-latency, privacy-preserving ways to triage infrastructure, water, sanitation, and health-risk reports from images and short notes.
2. Solution: Gemma4Good Field Report turns a captured image, time, location, and note into a structured civic report that a human can review before submission.
3. Gemma 4 usage: local Gemma 4 vision/instruction analysis through the same-origin API, with structured output aligned to schemas.
4. Architecture: browser capture workflow, Field Report API, private llama.cpp Vulkan Gemma 4 runtime, schema validation, human confirmation.
5. Technical choices: local-first deployment for privacy and resilience; structured JSON for reliable review; stale image-analysis guards in the UI; demo dataset marked separately from approved training data.
6. Impact: faster issue escalation for water quality, sanitation, infrastructure, and health-risk conditions in bandwidth-constrained contexts.
7. Evidence links: public repo/notebook, live demo, video, architecture notes, and reproducibility details.

In [ ]:
submission_summary = {
    "recommended_tracks": ["Impact Track: Global Resilience", "Impact Track: Safety & Trust", "Special Technology Track: llama.cpp"],
    "only_claim_unsloth_track_if": "a Gemma 4 fine-tuned model is completed, published, and documented",
    "rubric_mapping": {
        "Impact & Vision": "Show field reporting in low-connectivity/public-health infrastructure scenarios.",
        "Video Pitch & Storytelling": "Lead with a real field report, then show Gemma 4 turning it into a reviewable action report.",
        "Technical Depth & Execution": "Show local Gemma 4 runtime, API wiring, schemas, stale-analysis safeguards, and reproducible code.",
    },
    "links": submission_links,
}
print(json.dumps(submission_summary, indent=2))